# Selected-spectrum response to the LQCD priors

This notebook shows how each standardized PCA direction changes the selected $1\mu1p$ prediction. It includes only the **LQCD $k_{\max}=6$** and **MINERvA+LQCD $k_{\max}=6$** priors.

For every PCA component, the seven stored points ($-3\sigma$ through $+3\sigma$) are projected onto reconstructed $\log_{10}(Q^2/\mathrm{GeV}^2)$ with $p_n$ integrated out, and onto reconstructed $p_n$ with $Q^2$ integrated out. The lower panels show the ratio to the zero-sigma spectrum, making sub-percent changes visible.

The PCA branches contain **absolute weights relative to the original generator**, including the prior central value at zero sigma. Consequently the varied event weight is `non_genie_net_weight * PCA_sigma_weight`; the dedicated prior-CV branch is not multiplied a second time.

In [ ]:
from pathlib import Path

import awkward as ak
import matplotlib.pyplot as plt
import numpy as np
import uproot

plt.style.use("seaborn-v0_8-whitegrid")

INPUT_FILE = Path("/nevis/riverside/data/epelaez/ngem/intermediate_files/minimal_withspline_df.root")
TREE_NAME = "tree"
MC_POT = 9.57e19
TARGET_POT = 1.75e21
POT_SCALE = TARGET_POT / MC_POT
SIGMAS = np.arange(-3, 4)

Q2_EDGES = np.array([-2.00, -1.50, -1.20, -1.00, -0.85, -0.70, -0.55, -0.40, -0.20, 0.20])
PN_EDGES = np.array([0.00, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 1.00])

PRIORS = {
    "LQCD $k_{\max}=6$": {
        "cv": "weight_lqcd_k6_FA",
        "pca": ["weight_spline_FAzexpLQCDK6PCA1", "weight_spline_FAzexpLQCDK6PCA2"],
    },
    "MINERvA+LQCD $k_{\max}=6$": {
        "cv": "weight_minerva_lqcd_k6_FA",
        "pca": ["weight_spline_FAzexpMinervaLQCDK6PCA1", "weight_spline_FAzexpMinervaLQCDK6PCA2"],
    },
}

INPUT_FILE

## Load the selected overlay sample

This is the same nominal selection used by the prior-fit XMLs. Both true-$1\mu1p$ and background overlay subchannels are included, so integrating either projection gives the complete selected MC prediction.

In [ ]:
selection_branches = [
    "isdata", "isext", "isdirt", "isnuwro", "afro_1mu1p_sel",
    "afro_1mu1p_Q2", "afro_1mu1p_Pn", "non_genie_net_weight",
]
prior_branches = [name for prior in PRIORS.values() for name in [prior["cv"], *prior["pca"]]]

with uproot.open(INPUT_FILE) as root_file:
    events = root_file[TREE_NAME].arrays(selection_branches + prior_branches, library="ak")

selected = (
    (events.isdata == 0)
    & (events.isext == 0)
    & (events.isdirt == 0)
    & (events.isnuwro == 0)
    & (events.afro_1mu1p_sel == 1)
)
events = events[selected]
log10_q2 = np.log10(ak.to_numpy(events.afro_1mu1p_Q2))
pn = ak.to_numpy(events.afro_1mu1p_Pn)
base_weight = POT_SCALE * ak.to_numpy(events.non_genie_net_weight)

print(f"Selected overlay events: {len(events):,}")
print(f"POT scale: {POT_SCALE:.4f}")

## Build and validate the projected spectra

At zero sigma, every absolute PCA branch should reproduce the spectrum made with its dedicated prior-CV branch. The assertion below catches a missing central factor or accidental double counting before plotting.

In [ ]:
def histogram(values, edges, weights):
    return np.histogram(values, bins=edges, weights=weights)[0]


def projected_spectra(variable, edges, pca_weights):
    return np.stack([
        histogram(variable, edges, base_weight * pca_weights[:, i])
        for i in range(len(SIGMAS))
    ])


spectra = {}
for prior_label, prior in PRIORS.items():
    cv_weight = ak.to_numpy(events[prior["cv"]])
    cv_q2 = histogram(log10_q2, Q2_EDGES, base_weight * cv_weight)
    cv_pn = histogram(pn, PN_EDGES, base_weight * cv_weight)
    spectra[prior_label] = []

    for component, branch in enumerate(prior["pca"], start=1):
        pca_weights = ak.to_numpy(events[branch])
        q2_spectra = projected_spectra(log10_q2, Q2_EDGES, pca_weights)
        pn_spectra = projected_spectra(pn, PN_EDGES, pca_weights)
        np.testing.assert_allclose(q2_spectra[3], cv_q2, rtol=2e-5, atol=2e-5)
        np.testing.assert_allclose(pn_spectra[3], cv_pn, rtol=2e-5, atol=2e-5)
        spectra_2d = np.stack([
            np.histogram2d(
                log10_q2, pn, bins=(Q2_EDGES, PN_EDGES),
                weights=base_weight * pca_weights[:, i],
            )[0]
            for i in range(len(SIGMAS))
        ])
        spectra[prior_label].append({"q2": q2_spectra, "pn": pn_spectra, "2d": spectra_2d})

        total_ratios = q2_spectra.sum(axis=1) / q2_spectra[3].sum()
        changes = 100 * (total_ratios - 1)
        print(f"{prior_label}, PCA{component}: total-rate change at -1/+1 sigma = "
              f"{changes[2]:+.3f}% / {changes[4]:+.3f}%")

## Seven-point response for each PCA component

Each figure corresponds to one prior and one PCA component. The upper row contains the selected event spectra; the lower row is the bin-by-bin ratio to the CV line.

In [ ]:
sigma_colors = plt.cm.coolwarm(np.linspace(0.05, 0.95, len(SIGMAS)))


def sigma_label(sigma):
    if sigma == 0:
        return "CV"
    return rf"${sigma:+d}\sigma$"


def plot_component(prior_label, component, component_spectra):
    fig, axes = plt.subplots(
        2, 2, figsize=(13, 7.5), sharex="col",
        gridspec_kw={"height_ratios": [3, 1], "hspace": 0.08, "wspace": 0.22},
    )
    projections = [
        (component_spectra["q2"], Q2_EDGES, r"$\log_{10}(Q^2/\mathrm{GeV}^2)$"),
        (component_spectra["pn"], PN_EDGES, r"$p_n$ [GeV/$c$]"),
    ]

    for column, (values, edges, xlabel) in enumerate(projections):
        spectrum_ax, ratio_ax = axes[:, column]
        cv = values[3]
        for index, sigma in enumerate(SIGMAS):
            is_cv = sigma == 0
            color = "black" if is_cv else sigma_colors[index]
            width = 2.6 if is_cv else 1.7
            zorder = 10 if is_cv else 3
            spectrum_ax.stairs(values[index], edges, color=color, linewidth=width,
                               label=sigma_label(sigma), zorder=zorder)
            ratio = np.divide(values[index], cv, out=np.ones_like(cv), where=cv != 0)
            ratio_ax.stairs(ratio, edges, color=color, linewidth=width, zorder=zorder, baseline=None)

        spectrum_ax.set_ylabel("Selected events")
        spectrum_ax.set_title(f"Integrated over {'$p_n$' if column == 0 else '$Q^2$'}")
        valid_bins = cv > 0
        all_ratios = values[:, valid_bins] / cv[valid_bins]
        ratio_low = min(1.0, np.nanmin(all_ratios))
        ratio_high = max(1.0, np.nanmax(all_ratios))
        ratio_span = max(ratio_high - ratio_low, 0.002)
        ratio_ax.set_ylim(ratio_low - 0.12 * ratio_span, ratio_high + 0.12 * ratio_span)
        ratio_ax.axhline(1, color="0.45", linewidth=1, linestyle="--")
        ratio_ax.set_ylabel("/ CV")
        ratio_ax.set_xlabel(xlabel)
        ratio_ax.yaxis.set_major_formatter(plt.matplotlib.ticker.FormatStrFormatter("%.3f"))

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.subplots_adjust(top=0.88, right=0.82)
    fig.legend(handles, labels, loc="center left", bbox_to_anchor=(0.835, 0.5),
               ncol=1, frameon=False, title="PCA variation")
    fig.suptitle(f"{prior_label}: PCA{component} selected-spectrum response", y=0.97, fontsize=15)
    plt.show()


for prior_label, prior_spectra in spectra.items():
    for component, component_spectra in enumerate(prior_spectra, start=1):
        plot_component(prior_label, component, component_spectra)

## Fully sliced two-dimensional response

The first figure in each pair shows the bin-by-bin ratio to CV versus $p_n$, separately in every reconstructed $\log_{10}(Q^2)$ bin. The second shows the ratio to CV versus reconstructed $\log_{10}(Q^2)$, separately in every $p_n$ bin. Thus no analysis bin is integrated out. The CV is a horizontal line at one, and each panel is zoomed to its variation envelope. A dedicated legend column is reserved on the right of every figure.

In [ ]:
def draw_seven_ratio_lines(ax, values, edges):
    cv = values[3]
    valid_bins = cv > 0
    ratios = np.divide(values, cv, out=np.ones_like(values), where=cv != 0)
    for index, sigma in enumerate(SIGMAS):
        is_cv = sigma == 0
        ax.stairs(
            ratios[index], edges, baseline=None,
            color="black" if is_cv else sigma_colors[index],
            linewidth=2.4 if is_cv else 1.5,
            label=sigma_label(sigma),
            zorder=10 if is_cv else 3,
        )
    visible_ratios = ratios[:, valid_bins]
    ratio_low = min(1.0, np.nanmin(visible_ratios))
    ratio_high = max(1.0, np.nanmax(visible_ratios))
    ratio_span = max(ratio_high - ratio_low, 0.002)
    ax.set_ylim(ratio_low - 0.12 * ratio_span, ratio_high + 0.12 * ratio_span)
    ax.axhline(1, color="0.45", linewidth=1, linestyle="--", zorder=1)
    ax.yaxis.set_major_formatter(plt.matplotlib.ticker.FormatStrFormatter("%.3f"))


def finish_sliced_figure(fig, axes, prior_label, component, slice_description):
    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.subplots_adjust(top=0.88, right=0.84, hspace=0.36, wspace=0.28)
    fig.legend(
        handles, labels, loc="center left", bbox_to_anchor=(0.855, 0.5),
        ncol=1, frameon=False, title="PCA variation",
    )
    fig.suptitle(
        f"{prior_label}: PCA{component} — {slice_description}",
        y=0.97, fontsize=15,
    )
    plt.show()


def plot_pn_in_q2_slices(prior_label, component, spectra_2d):
    fig, axes = plt.subplots(3, 3, figsize=(15, 10), sharex=True)
    for q2_bin, ax in enumerate(axes.flat):
        draw_seven_ratio_lines(ax, spectra_2d[:, q2_bin, :], PN_EDGES)
        low, high = Q2_EDGES[q2_bin:q2_bin + 2]
        ax.set_title(rf"${low:.2f} \leq \log_{{10}}(Q^2) < {high:.2f}$", fontsize=10)
        ax.set_xlabel(r"$p_n$ [GeV/$c$]")
        ax.set_ylabel("Prediction / CV")
    finish_sliced_figure(fig, axes, prior_label, component,
                         r"$p_n$ in each $\log_{10}(Q^2)$ bin")


def plot_q2_in_pn_slices(prior_label, component, spectra_2d):
    fig, axes = plt.subplots(2, 4, figsize=(15, 7.5), sharex=True)
    for pn_bin, ax in enumerate(axes.flat):
        draw_seven_ratio_lines(ax, spectra_2d[:, :, pn_bin], Q2_EDGES)
        low, high = PN_EDGES[pn_bin:pn_bin + 2]
        ax.set_title(rf"${low:.2f} \leq p_n < {high:.2f}$ GeV/$c$", fontsize=10)
        ax.set_xlabel(r"$\log_{10}(Q^2/\mathrm{GeV}^2)$")
        ax.set_ylabel("Prediction / CV")
    finish_sliced_figure(fig, axes, prior_label, component,
                         r"$\log_{10}(Q^2)$ in each $p_n$ bin")


for prior_label, prior_spectra in spectra.items():
    for component, component_spectra in enumerate(prior_spectra, start=1):
        plot_pn_in_q2_slices(prior_label, component, component_spectra["2d"])
        plot_q2_in_pn_slices(prior_label, component, component_spectra["2d"])